# Day 01 — Exercises
**Complete these before moving to Day 02.**  
Solutions are in the final cell — but attempt each exercise first!

---


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

spark = (
    SparkSession.builder
    .appName("day01_exercises")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Ready")


---
### Exercise 1 — Lazy evaluation chain
Create a DataFrame of integers 1–50.  
Filter to keep only numbers divisible by 3.  
Add a column `cubed` = the cube of each number.  
**Print the count first, then show the result.**  
Expected: 16 rows (3, 6, 9, … 48).


In [ ]:
# Your code here


---
### Exercise 2 — GroupBy + aggregation
Using the employees data below, compute per-city: average salary, max salary, and headcount.  
Order by average salary descending.


In [ ]:
employees = spark.createDataFrame([
    ("Alice",   "Toronto",  95_000),
    ("Bob",     "Montreal", 72_000),
    ("Carol",   "Toronto",  88_000),
    ("Dave",    "Calgary",  81_000),
    ("Eve",     "Montreal", 102_000),
    ("Frank",   "Calgary",  76_000),
    ("Grace",   "Toronto",  91_000),
], schema=["name", "city", "salary"])

# Your code here


---
### Exercise 3 — Partitions
Using the employees DataFrame above:
1. Print its current partition count
2. Repartition to 4 partitions and print the count
3. Coalesce to 2 and print the count
4. In a comment: which of these caused a shuffle and why?


In [ ]:
# Your code here


---
### Exercise 4 — Conceptual (answer in comments)
Given this code:
```python
df = spark.range(1_000_000)
df2 = df.filter(df.id > 500_000)
df3 = df2.withColumn("doubled", df2.id * 2)
df4 = df3.groupBy((df3.doubled % 10).alias("last_digit")).count()
df4.show()
```
Answer in comments below:
- Q1: Which line(s) actually trigger computation?
- Q2: Classify each transformation as narrow or wide: filter, withColumn, groupBy, count
- Q3: How many stage boundaries (shuffles) do you expect? How many stages total?


In [ ]:
# Q1: 
# Q2: filter= , withColumn= , groupBy= , count=
# Q3: stage boundaries= , total stages=


---
### Exercise 5 — Explicit schema
Define an explicit `StructType` schema for this data and create a DataFrame with it.  
Then print the schema and show the result.
```
product_id | product_name    | price  | in_stock
1          | Spark Book      | 49.99  | True
2          | Kafka Guide     | 44.99  | True  
3          | Flink Handbook  | 39.99  | False
```


In [ ]:
# Your code here


---
## Solutions (expand after attempting)


In [ ]:
# ── SOLUTION 1 ────────────────────────────────────────────────────────────────
divisible_by_3 = spark.range(1, 51).filter(F.col("id") % 3 == 0)
with_cubed = divisible_by_3.withColumn("cubed", F.col("id") ** 3)
print(f"Count: {with_cubed.count()}")
with_cubed.show()

# ── SOLUTION 2 ────────────────────────────────────────────────────────────────
(employees
 .groupBy("city")
 .agg(
     F.round(F.avg("salary"), 2).alias("avg_salary"),
     F.max("salary").alias("max_salary"),
     F.count("*").alias("headcount")
 )
 .orderBy("avg_salary", ascending=False)
 .show())

# ── SOLUTION 3 ────────────────────────────────────────────────────────────────
print(f"Original:       {employees.rdd.getNumPartitions()} partitions")
r4 = employees.repartition(4)
print(f"repartition(4): {r4.rdd.getNumPartitions()} partitions  ← full shuffle (can go up or down)")
c2 = r4.coalesce(2)
print(f"coalesce(2):    {c2.rdd.getNumPartitions()} partitions  ← no shuffle (merge local only, can only go down)")

# ── SOLUTION 4 ────────────────────────────────────────────────────────────────
# Q1: Only df4.show() triggers computation (it is the action)
# Q2: filter=narrow, withColumn=narrow, groupBy=wide, count=action(wide)
# Q3: 1 shuffle (at groupBy), 2 stages total (before shuffle, after shuffle)

# ── SOLUTION 5 ────────────────────────────────────────────────────────────────
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, BooleanType

product_schema = StructType([
    StructField("product_id",   IntegerType(), nullable=False),
    StructField("product_name", StringType(),  nullable=False),
    StructField("price",        DoubleType(),  nullable=False),
    StructField("in_stock",     BooleanType(), nullable=False),
])

products = spark.createDataFrame([
    (1, "Spark Book",     49.99, True),
    (2, "Kafka Guide",    44.99, True),
    (3, "Flink Handbook", 39.99, False),
], schema=product_schema)

products.printSchema()
products.show()


In [ ]:
spark.stop()
